In [0]:
storage_account = "vesseldatavd99"  # e.g., vesseldatavd99
storage_key = "TUqbHBJ5+ZvfOHAXTGwH75HDC9QAcAqMXinMWhfAa3Xo9rSwnkQyOzRBAAPVtToDpGnT+ZudKdII+AStmezehA=="  # key1 from your Azure storage account

# Set credentials directly in the Spark session
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key
)
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net", storage_key
)

In [0]:
raw_container_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/"

# If your storage account does NOT have Hierarchical Namespace enabled, use wasbs:
# raw_container_path = f"wasbs://raw@{storage_account}.blob.core.windows.net/"

display(dbutils.fs.ls(raw_container_path))

path,name,size,modificationTime
abfss://raw@vesseldatavd99.dfs.core.windows.net/ais_data.csv,ais_data.csv,876791232,1791494060000


In [0]:
file_path = "abfss://raw@vesseldatavd99.dfs.core.windows.net/ais_data.csv"

# Read the raw CSV into a PySpark DataFrame
raw_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(file_path)
)

print(f"Total raw records: {raw_df.count():,}")
raw_df.printSchema()
display(raw_df.limit(10))

Total raw records: 8,156,509
root
 |-- MMSI: integer (nullable = true)
 |-- BaseDateTime: timestamp (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- SOG: double (nullable = true)
 |-- COG: double (nullable = true)
 |-- Heading: double (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: integer (nullable = true)
 |-- Status: integer (nullable = true)
 |-- Length: integer (nullable = true)
 |-- Width: integer (nullable = true)
 |-- Draft: double (nullable = true)
 |-- Cargo: integer (nullable = true)
 |-- TransceiverClass: string (nullable = true)



MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass
368926035,2023-01-01T00:00:05Z,38.65165,-90.17964,0.1,360.0,511.0,KIMMSWICK,null,AENA,33,15,null,null,null,33,A
367647050,2023-01-01T00:00:04Z,30.16506,-90.99936,6.3,161.4,511.0,USS CAIRO,null,WDH7325,31,0,20,null,null,31,A
352001704,2023-01-01T00:00:04Z,28.91963,-94.37033,14.7,97.5,97.0,ENEOS EXPLORER,IMO9935492,3E2723,80,0,230,36,11.7,80,A
367104060,2023-01-01T00:00:00Z,34.33537,-119.56046,14.8,202.9,511.0,ALAN T,IMO8982577,WDC9571,60,15,28,7,null,0,A
367099730,2023-01-01T00:00:02Z,26.53889,-97.40485,0.0,360.0,511.0,JESSIE M,null,WDJ2892,52,0,0,0,0.0,52,A
367579120,2023-01-01T00:00:03Z,29.32348,-94.78112,0.0,248.5,511.0,TERRY FONTENOT,null,WDG8590,31,0,16,7,null,52,A
367015890,2023-01-01T00:00:09Z,39.853,-75.24165,0.0,360.0,511.0,DOROTHY J,null,WDC3893,31,0,19,7,null,31,A
366995260,2023-01-01T00:00:01Z,30.47154,-91.19402,0.6,166.3,183.0,CAPT JAMES HOOVER,null,WDC2583,31,0,16,6,null,52,A
538008717,2023-01-01T00:00:01Z,23.64963,-83.18562,5.7,269.0,274.0,VELOS FORTUNA,IMO9347310,V7A2686,89,0,182,32,7.5,89,A
368261060,2023-01-01T00:00:06Z,30.51808,-91.22525,9.8,246.6,246.0,AMERICAN SYMPHONY,null,null,60,0,100,18,0.0,60,A


In [0]:
import datetime
import io
import zipfile
import requests
from azure.storage.filedatalake import DataLakeServiceClient
from pyspark.sql import functions as F

# 1. Storage credentials
storage_account = "vesseldatavd99"
storage_key = "TUqbHBJ5+ZvfOHAXTGwH75HDC9QAcAqMXinMWhfAa3Xo9rSwnkQyOzRBAAPVtToDpGnT+ZudKdII+AStmezehA=="  # key1 from your Azure storage account

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key
)

# Initialize Azure ADLS Gen2 client
service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account}.dfs.core.windows.net",
    credential=storage_key,
)
raw_fs_client = service_client.get_file_system_client("raw")

bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/ais_bronze"

# 2. Date range for 2019
start_date = datetime.date(2019, 1, 1)
dates_to_pull = [start_date + datetime.timedelta(days=i) for i in range(7)]

# 3. Stream NOAA -> Upload to raw abfss:// -> Read into Spark -> Append Bronze Delta
for single_date in dates_to_pull:
    date_str = single_date.strftime("%Y_%m_%d")
    zip_url = f"https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2019/AIS_{date_str}.zip"
    blob_name = f"AIS_{date_str}.csv"
    raw_abfss_path = (
        f"abfss://raw@{storage_account}.dfs.core.windows.net/{blob_name}"
    )

    print(f"Fetching {zip_url}...")
    res = requests.get(zip_url, stream=True)

    if res.status_code == 200:
        # Stream unpack in memory
        with zipfile.ZipFile(io.BytesIO(res.content)) as z:
            csv_names = [f for f in z.namelist() if f.endswith(".csv")]
            if csv_names:
                print(f"Uploading {blob_name} to raw container...")
                file_client = raw_fs_client.get_file_client(blob_name)
                file_client.upload_data(z.read(csv_names[0]), overwrite=True)

                # Spark reads directly from ADLS Gen2, bypassing driver filesystem restrictions
                df_day = (
                    spark.read.format("csv")
                    .option("header", "true")
                    .option("inferSchema", "true")
                    .load(raw_abfss_path)
                    .withColumn("ingestion_timestamp", F.current_timestamp())
                    .withColumn(
                        "source_file_date", F.to_date(F.lit(single_date))
                    )
                )

                # Append to Bronze Delta table
                (
                    df_day.write.format("delta")
                    .mode("append")
                    .option("mergeSchema", "true")
                    .save(bronze_path)
                )

                print(
                    f"Successfully ingested {df_day.count():,} rows for {date_str} to Bronze Delta."
                )
    else:
        print(f"Skipping {date_str}: HTTP {res.status_code}")

# 4. Total record validation
bronze_df = spark.read.format("delta").load(bronze_path)
print(f"\nTotal Bronze records across all 7 days: {bronze_df.count():,}")

Fetching https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2019/AIS_2019_01_01.zip...
Uploading AIS_2019_01_01.csv to raw container...
Successfully ingested 7,516,408 rows for 2019_01_01 to Bronze Delta.
Fetching https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2019/AIS_2019_01_02.zip...
Uploading AIS_2019_01_02.csv to raw container...
Successfully ingested 7,169,185 rows for 2019_01_02 to Bronze Delta.
Fetching https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2019/AIS_2019_01_03.zip...
Uploading AIS_2019_01_03.csv to raw container...
Successfully ingested 6,724,018 rows for 2019_01_03 to Bronze Delta.
Fetching https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2019/AIS_2019_01_04.zip...
Uploading AIS_2019_01_04.csv to raw container...
Successfully ingested 6,951,276 rows for 2019_01_04 to Bronze Delta.
Fetching https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2019/AIS_2019_01_05.zip...
Uploading AIS_2019_01_05.csv to raw container...
Successfully ingested 6,851,182 rows for 2019_01_

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:728)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:446)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:446)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can